In [1]:
import requests
import pandas as pd
import time

all_subfields = []
page = 1

while True:
    url = f"https://api.openalex.org/subfields?per_page=200&page={page}"
    response = requests.get(url)
    
    if response.status_code != 200:
        break
        
    data = response.json().get('results', [])
    if not data:
        break
        
    all_subfields.extend(data)
    print(f"  - Fetched page {page} ({len(data)} subfields)")
    
    page += 1
    time.sleep(0.5) 

hierarchy_map = {}
for sf in all_subfields:
    sf_id = sf['id'].split('/')[-1] # e.g., "1100"
    
    field = sf.get('field', {})
    domain = sf.get('domain', {})
    
    hierarchy_map[sf_id] = {
        'Subfield': sf['display_name'],
        'ID': field.get('id', '').split('/')[-1],
        'Field': field.get('display_name', ''),
        'Domain': domain.get('display_name', '')
    }

print(f"Successfully built hierarchy for {len(hierarchy_map)} subfields.")
pd.DataFrame.from_dict(hierarchy_map, orient='index').head()


  - Fetched page 1 (200 subfields)
  - Fetched page 2 (52 subfields)
Successfully built hierarchy for 252 subfields.


,Subfield,ID,Field,Domain
2202,Aerospace Engineering,22,Engineering,Physical Sciences
3312,Sociology and Political Science,33,Social Sciences,Social Sciences
3106,Nuclear and High Energy Physics,31,Physics and Astronomy,Physical Sciences
1312,Molecular Biology,13,"Biochemistry, Genetics and Molecular Biology",Life Sciences
2002,Economics and Econometrics,20,"Economics, Econometrics and Finance",Social Sciences


In [ ]:
import collections
import datetime

current_year = 2025
start_year = 1899
year_range = f"{start_year}-{current_year}"  

print(f"Fetching data for years: {year_range}")
uva_id = "I51556381"
select_fields = "id,primary_topic"
filter_str = f"authorships.institutions.lineage:{uva_id},publication_year:{year_range}"

cursor = "*"
subfield_counts = collections.Counter()
total_works = 0

while True:
    url = (
        f"https://api.openalex.org/works?"
        f"filter={filter_str}"
        f"&select={select_fields}"
        f"&per_page=200&cursor={cursor}"
    )
    
    try:
        r = requests.get(url)
        r.raise_for_status() # Raise error for 400/500 codes
        data = r.json()
        
        results = data.get('results', [])
        if not results:
            break
            
        cursor = data['meta']['next_cursor']
        
        # Count the subfields in this batch
        for work in results:
            topic = work.get('primary_topic')
            if topic and topic.get('subfield'):
                sf_id = topic['subfield']['id'].split('/')[-1]
                subfield_counts[sf_id] += 1
        
        total_works += len(results)
        if total_works % 10000 == 0:
            print(f"  - Processed {total_works} works...")
        
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

print(f"Finished. Analyzed {total_works} papers across {len(subfield_counts)} subfields.")

Fetching data for years: 2016-2026
  - Processed 10000 works...
  - Processed 20000 works...
  - Processed 30000 works...
  - Processed 40000 works...
  - Processed 50000 works...
  - Processed 60000 works...
  - Processed 70000 works...
Finished. Analyzed 73871 papers across 245 subfields.


In [3]:

rows = []
for sf_id, count in subfield_counts.items():
    info = hierarchy_map.get(sf_id)
    if info:
        rows.append({
            'Domain': info['Domain'],
            'Group': info['Field'],
            'Field': info['Subfield'],
            'numpub': count,
        })

df = pd.DataFrame(rows)

# Sort Hierarchy: Domain -> Group -> Field
df = df.sort_values(by=['Domain', 'Group', 'Field'])

print(f"\nPreview of Results ({len(df)} rows):")
df.head()


Preview of Results (245 rows):


,Domain,Group,Field,numpub
217,Health Sciences,Dentistry,General Dentistry,4
227,Health Sciences,Dentistry,Oral Surgery,6
132,Health Sciences,Dentistry,Orthodontics,3
154,Health Sciences,Dentistry,Periodontics,26
233,Health Sciences,Health Professions,Complementary and Manual Therapy,4


In [4]:


df = df.sort_values(by=['Domain', 'Group', 'Field'])
df = df[['Domain', 'Group', 'Field', 'numpub']]

print(f"\nPreview of Results ({len(df)} rows):")
df.head()


Preview of Results (245 rows):


,Domain,Group,Field,numpub
217,Health Sciences,Dentistry,General Dentistry,4
227,Health Sciences,Dentistry,Oral Surgery,6
132,Health Sciences,Dentistry,Orthodontics,3
154,Health Sciences,Dentistry,Periodontics,26
233,Health Sciences,Health Professions,Complementary and Manual Therapy,4


In [5]:
# Export to CSV
df.to_csv("uva_2025_full_hierarchy.csv", index=False)